# Dataset Exploration

This notebook explores the three main datasets used for protein LLM post-training:

1. **IPD PDB Sample** - Protein structures in PyTorch format
2. **Swiss-Prot** - Curated protein sequences (FASTA)
3. **Mol-Instructions** - Instruction-following pairs for protein tasks

In [ ]:
import os
import json
import gzip
from pathlib import Path
import torch
import pandas as pd
from collections import Counter

# Set data directory
DATA_DIR = Path("../data/raw")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Contents: {list(DATA_DIR.iterdir())}")

Data directory: /orcd/pool/006/yeopjin/workspace/Post_Training_Protein_LLM/data/raw
Contents: [PosixPath('../data/raw/pdb_2021aug02_sample'), PosixPath('../data/raw/swissprot'), PosixPath('../data/raw/mol_instructions'), PosixPath('../data/raw/pdb_files')]


---
## 1. IPD PDB Sample Dataset

Pre-processed protein structures from the Protein Data Bank in PyTorch tensor format.
Used by RoseTTAFold/ProteinMPNN for structure-based tasks.

In [ ]:
pdb_dir = DATA_DIR / "pdb_2021aug02_sample"
print(f"IPD PDB Sample directory: {pdb_dir}")
print(f"\nDirectory contents:")
for item in pdb_dir.iterdir():
    if item.is_file():
        print(f"  {item.name} ({item.stat().st_size / 1024 / 1024:.2f} MB)")
    else:
        print(f"  {item.name}/")

IPD PDB Sample directory: ../data/raw/pdb_2021aug02_sample

Directory contents:
  README (0.00 MB)
  valid_clusters.txt (0.01 MB)
  pdb/
  test_clusters.txt (0.01 MB)
  list.csv (167.78 MB)


In [ ]:
# Read the README
readme_path = pdb_dir / "README"
if readme_path.exists():
    print("=" * 60)
    print("README Contents:")
    print("=" * 60)
    print(readme_path.read_text())

README Contents:

Training set for RosettaFold/MPNN etc.

Each PDB entry is represented as a collection of .pt files:
    PDBID_CHAINID.pt - contains CHAINID chain from PDBID
    PDBID.pt         - metadata and information on biological assemblies

PDBID_CHAINID.pt has the following fields:
    seq  - amino acid sequence (string)
    xyz  - atomic coordinates [L,14,3]
    mask - boolean mask [L,14]
    bfac - temperature factors [L,14]
    occ  - occupancy [L,14] (is 1 for most atoms, <1 if alternative conformations are present)

PDBID.pt:
    method        - experimental method (str)
    date          - deposition date (str)
    resolution    - resolution (float)
    chains        - list of CHAINIDs (there is a corresponding PDBID_CHAINID.pt file for each of these)
    tm            - pairwise similarity between chains (TM-score,seq.id.,rmsd from TM-align) [num_chains,num_chains,3]
    asmb_ids      - biounit IDs as in the PDB (list of str)
    asmb_details  - how the assembly was ide

In [4]:
# Count .pt files
pt_files = list(pdb_dir.glob("**/*.pt"))
print(f"Total .pt files: {len(pt_files)}")
print(f"\nFirst 10 files:")
for f in pt_files[:10]:
    print(f"  {f.name}")

Total .pt files: 870

First 10 files:
  5l3o_B.pt
  5l3z_A.pt
  3l3i_C.pt
  4l3g_D.pt
  6l3q_B.pt
  7l3e.pt
  6l33_C.pt
  2l3d_A.pt
  6l35_I.pt
  6l3t_F.pt


In [5]:
# Load and inspect a sample .pt file
sample_pt = pt_files[0]
print(f"Loading sample file: {sample_pt.name}")
print("=" * 60)

data = torch.load(sample_pt, map_location='cpu', weights_only=False)
print(f"Type: {type(data)}")

if isinstance(data, dict):
    print(f"\nKeys: {list(data.keys())}")
    print("\nContents:")
    for key, value in data.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: Tensor shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, str):
            print(f"  {key}: '{value[:100]}...'" if len(value) > 100 else f"  {key}: '{value}'")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print(f"Data: {data}")

Loading sample file: 5l3o_B.pt


Type: <class 'dict'>

Keys: ['seq', 'xyz', 'mask', 'bfac', 'occ']

Contents:
  seq: 'MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPLSVSYDQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSL...'
  xyz: Tensor shape=torch.Size([260, 14, 3]), dtype=torch.float32
  mask: Tensor shape=torch.Size([260, 14]), dtype=torch.float32
  bfac: Tensor shape=torch.Size([260, 14]), dtype=torch.float32
  occ: Tensor shape=torch.Size([260, 14]), dtype=torch.float32


In [6]:
# Load the list.csv for metadata
list_csv = pdb_dir / "list.csv"
if list_csv.exists():
    print("Loading list.csv (first 1000 rows)...")
    df = pd.read_csv(list_csv, nrows=1000)
    print(f"\nShape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst 5 entries:")
    display(df.head())
    print("\nDataset statistics:")
    display(df.describe())

Loading list.csv (first 1000 rows)...



Shape: (1000, 6)
Columns: ['CHAINID', 'DEPOSITION', 'RESOLUTION', 'HASH', 'CLUSTER', 'SEQUENCE']

First 5 entries:


,CHAINID,DEPOSITION,RESOLUTION,HASH,CLUSTER,SEQUENCE
0,5naf_A,2017-02-27,2.493,57113,12123,MGSSHHHHHHSSGLEVLFQGPEENGAHTIANNHTDMMEVDGDVEIP...
1,5naf_B,2017-02-27,2.493,57113,12123,MGSSHHHHHHSSGLEVLFQGPEENGAHTIANNHTDMMEVDGDVEIP...
2,5naf_C,2017-02-27,2.493,57113,12123,MGSSHHHHHHSSGLEVLFQGPEENGAHTIANNHTDMMEVDGDVEIP...
3,5naf_D,2017-02-27,2.493,57113,12123,MGSSHHHHHHSSGLEVLFQGPEENGAHTIANNHTDMMEVDGDVEIP...
4,5nag_A,2017-02-27,1.680,82310,6750,MTATDNARQVTIIGAGLAGTLVARLLARNGWQVNLFERRPDPRIET...



Dataset statistics:


,RESOLUTION,HASH,CLUSTER
count,1000.000000,1000.000000,1000.000000
mean,2.408139,56330.524000,14108.498000
std,0.744853,28382.493403,8929.962557
min,1.010000,419.000000,52.000000
25%,1.900000,34274.000000,5051.000000
50%,2.250000,54794.000000,14138.000000
75%,2.850000,75129.000000,23334.000000
max,4.800000,113754.000000,27699.000000


---
## 2. Swiss-Prot Dataset

Curated protein sequences from UniProt in FASTA format.
Contains ~570K reviewed protein sequences with annotations.

In [ ]:
swissprot_dir = DATA_DIR / "swissprot"
fasta_gz = swissprot_dir / "uniprot_sprot.fasta.gz"

print(f"Swiss-Prot file: {fasta_gz}")
print(f"File size: {fasta_gz.stat().st_size / 1024 / 1024:.2f} MB (compressed)")

Swiss-Prot file: ../data/raw/swissprot/uniprot_sprot.fasta.gz
File size: 89.13 MB (compressed)


In [ ]:
def parse_fasta_gz(filepath, max_entries=1000):
    """Parse gzipped FASTA file and return list of (header, sequence) tuples."""
    entries = []
    with gzip.open(filepath, 'rt') as f:
        header = None
        seq_lines = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header is not None:
                    entries.append((header, ''.join(seq_lines)))
                    if len(entries) >= max_entries:
                        break
                header = line[1:]  # Remove '>'
                seq_lines = []
            else:
                seq_lines.append(line)
        if header is not None and len(entries) < max_entries:
            entries.append((header, ''.join(seq_lines)))
    return entries

print("Parsing Swiss-Prot FASTA (first 1000 entries)...")
swissprot_entries = parse_fasta_gz(fasta_gz, max_entries=1000)
print(f"Loaded {len(swissprot_entries)} entries")

Parsing Swiss-Prot FASTA (first 1000 entries)...
Loaded 1000 entries


In [9]:
# Display sample entries
print("Sample Swiss-Prot entries:")
print("=" * 60)
for i, (header, seq) in enumerate(swissprot_entries[:5]):
    print(f"\n[Entry {i+1}]")
    print(f"Header: {header[:100]}..." if len(header) > 100 else f"Header: {header}")
    print(f"Sequence length: {len(seq)} amino acids")
    print(f"Sequence (first 80 aa): {seq[:80]}...")

Sample Swiss-Prot entries:

[Entry 1]
Header: sp|Q6GZX4|001R_FRG3G Putative transcription factor 001R OS=Frog virus 3 (isolate Goorha) OX=654924 G...
Sequence length: 256 amino acids
Sequence (first 80 aa): MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPPRVQVECPKAPVEWNNPPSEKGLIVGHFSGIKYKGEKAQ...

[Entry 2]
Header: sp|Q6GZX3|002L_FRG3G Uncharacterized protein 002L OS=Frog virus 3 (isolate Goorha) OX=654924 GN=FV3-...
Sequence length: 320 amino acids
Sequence (first 80 aa): MSIIGATRLQNDKSDTYSAGPCYAGGCSAFTPRGTCGKDWDLGEQTCASGFCTSQPLCARIKKTQVCGLRYSSKGKDPLV...

[Entry 3]
Header: sp|Q197F8|002R_IIV3 Uncharacterized protein 002R OS=Invertebrate iridescent virus 3 OX=345201 GN=IIV...
Sequence length: 458 amino acids
Sequence (first 80 aa): MASNTVSAQGGSNRPVRDFSNIQDVAQFLLFDPIWNEQPGSIVPWKMNREQALAERYPELQTSEPSEDYSGPVESLELLP...

[Entry 4]
Header: sp|Q197F7|003L_IIV3 Uncharacterized protein 003L OS=Invertebrate iridescent virus 3 OX=345201 GN=IIV...
Sequence length: 156 amino acids
Sequence (first 80 aa): M

In [10]:
# Sequence length distribution
seq_lengths = [len(seq) for _, seq in swissprot_entries]
print("Sequence Length Statistics (first 1000 entries):")
print(f"  Min: {min(seq_lengths)}")
print(f"  Max: {max(seq_lengths)}")
print(f"  Mean: {sum(seq_lengths)/len(seq_lengths):.1f}")
print(f"  Median: {sorted(seq_lengths)[len(seq_lengths)//2]}")

# Amino acid composition
all_aa = ''.join(seq for _, seq in swissprot_entries)
aa_counts = Counter(all_aa)
print(f"\nAmino Acid Composition:")
for aa, count in aa_counts.most_common(10):
    print(f"  {aa}: {count:,} ({100*count/len(all_aa):.2f}%)")

Sequence Length Statistics (first 1000 entries):
  Min: 10
  Max: 1707
  Mean: 298.4
  Median: 260

Amino Acid Composition:
  L: 28,608 (9.59%)
  A: 21,307 (7.14%)
  E: 21,142 (7.09%)
  S: 21,088 (7.07%)
  K: 18,507 (6.20%)
  V: 18,504 (6.20%)
  G: 17,232 (5.78%)
  I: 16,975 (5.69%)
  D: 16,662 (5.58%)
  R: 15,890 (5.33%)


In [11]:
# Parse header for species and protein info
def parse_swissprot_header(header):
    """Parse Swiss-Prot FASTA header.
    Format: sp|ACCESSION|ENTRY_NAME PROTEIN_NAME OS=ORGANISM OX=TAXID GN=GENE PE=EVIDENCE SV=VERSION
    """
    parts = header.split(' ', 1)
    ids = parts[0].split('|')
    accession = ids[1] if len(ids) > 1 else 'N/A'
    entry_name = ids[2] if len(ids) > 2 else 'N/A'
    
    description = parts[1] if len(parts) > 1 else ''
    
    # Extract organism
    organism = 'N/A'
    if 'OS=' in description:
        start = description.index('OS=') + 3
        end = description.find(' OX=', start) if ' OX=' in description else len(description)
        organism = description[start:end]
    
    return {'accession': accession, 'entry_name': entry_name, 'organism': organism}

# Create DataFrame
swissprot_data = []
for header, seq in swissprot_entries:
    info = parse_swissprot_header(header)
    info['sequence_length'] = len(seq)
    info['sequence'] = seq[:50] + '...' if len(seq) > 50 else seq
    swissprot_data.append(info)

swissprot_df = pd.DataFrame(swissprot_data)
print("Swiss-Prot entries as DataFrame:")
display(swissprot_df.head(10))

Swiss-Prot entries as DataFrame:


,accession,entry_name,organism,sequence_length,sequence
0,Q6GZX4,001R_FRG3G,Frog virus 3 (isolate Goorha),256,MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPPRVQV...
1,Q6GZX3,002L_FRG3G,Frog virus 3 (isolate Goorha),320,MSIIGATRLQNDKSDTYSAGPCYAGGCSAFTPRGTCGKDWDLGEQT...
2,Q197F8,002R_IIV3,Invertebrate iridescent virus 3,458,MASNTVSAQGGSNRPVRDFSNIQDVAQFLLFDPIWNEQPGSIVPWK...
3,Q197F7,003L_IIV3,Invertebrate iridescent virus 3,156,MYQAINPCPQSWYGSPQLEREIVCKMSGAPHYPNYYPVHPNALGGA...
4,Q6GZX2,003R_FRG3G,Frog virus 3 (isolate Goorha),438,MARPLLGKTSSVRRRLESLSACSIFFFLRKFCQKMASLVFLNSPVY...
5,Q6GZX1,004R_FRG3G,Frog virus 3 (isolate Goorha),60,MNAKYDTDQGVGRMLFLGTIGLAVVVGGLMAYGYYYDGKTPSSGTS...
6,Q197F5,005L_IIV3,Invertebrate iridescent virus 3,217,MRYTVLIALQGALLLLLLIDDGQGQSPYPYPGMPCNSSRQCGLGTC...
7,Q6GZX0,005R_FRG3G,Frog virus 3 (isolate Goorha),204,MQNPLPEVMSPEHDKRTTTPMSKEANKFIRELDKKPGDLAVVSDFV...
8,Q91G88,006L_IIV6,Invertebrate iridescent virus 6,352,MDSLNEVCYEQIKGTFYKGLFGDFPLIVDKKTGCFNATKLCVLGGK...
9,Q6GZW9,006R_FRG3G,Frog virus 3 (isolate Goorha),75,MYKMYFLKDQKFSLSGTIRINDKTQSEYGSVWCPGLSITGLHHDAI...


In [12]:
# Top organisms
print("Top 10 Organisms:")
print(swissprot_df['organism'].value_counts().head(10))

Top 10 Organisms:


organism
Invertebrate iridescent virus 6                                                116
Frog virus 3 (isolate Goorha)                                                   83
Arabidopsis thaliana                                                            51
Invertebrate iridescent virus 3                                                 47
African swine fever virus (isolate Pig/Kenya/KEN-50/1950)                       33
African swine fever virus (isolate Tick/Malawi/Lil 20-1/1983)                   32
African swine fever virus (isolate Tick/South Africa/Pretoriuskop Pr4/1996)     32
Mus musculus                                                                    32
African swine fever virus (isolate Warthog/Namibia/Wart80/1980)                 31
Homo sapiens                                                                    23
Name: count, dtype: int64


---
## 3. Mol-Instructions Dataset

Instruction-following pairs for protein-related tasks.
Contains multiple task types: protein design, function prediction, catalytic activity, etc.

In [13]:
mol_inst_dir = DATA_DIR / "mol_instructions" / "data" / "Protein-oriented_Instructions"
print(f"Mol-Instructions directory: {mol_inst_dir}")
print("\nAvailable files:")

json_files = list(mol_inst_dir.glob("*.json"))
for f in json_files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")

Mol-Instructions directory: ../data/raw/mol_instructions/data/Protein-oriented_Instructions

Available files:
  general_function.json: 121.50 MB
  protein_function.json: 147.49 MB
  domain_motif.json: 47.68 MB
  catalytic_activity.json: 61.90 MB
  protein_design.json: 268.22 MB


In [14]:
def load_json_samples(filepath, max_samples=100):
    """Load samples from a JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data[:max_samples]
    return data

# Load samples from each file
mol_instructions_data = {}
for json_file in json_files:
    task_name = json_file.stem
    print(f"Loading {task_name}...")
    samples = load_json_samples(json_file, max_samples=100)
    mol_instructions_data[task_name] = samples
    print(f"  Loaded {len(samples)} samples")

Loading general_function...


  Loaded 100 samples
Loading protein_function...


  Loaded 100 samples
Loading domain_motif...


  Loaded 100 samples
Loading catalytic_activity...


  Loaded 100 samples
Loading protein_design...


  Loaded 100 samples


In [15]:
# Explore each task type
for task_name, samples in mol_instructions_data.items():
    print("=" * 60)
    print(f"Task: {task_name}")
    print("=" * 60)
    
    if samples:
        sample = samples[0]
        print(f"Fields: {list(sample.keys())}")
        print(f"\nSample entry:")
        for key, value in sample.items():
            if isinstance(value, str):
                display_val = value[:200] + '...' if len(value) > 200 else value
                print(f"  {key}: {display_val}")
            else:
                print(f"  {key}: {value}")
    print()

Task: general_function
Fields: ['instruction', 'input', 'output', 'metadata']

Sample entry:
  instruction: Inspect the protein with the subsequent sequence and offer a concise description of its properties: 
  input: ```
MTDKKFNVAVVGATGAVGETMLSILEQRHFPVGEVYPLASSRSAGKRIEFNNKQLIVEDLETFDFSKVQIGLFSPGASVSAIHAPRAVEAGCVVVDNTSQFRYDDDIPLVVSEVNPHAVADYKNRGIIANPNCSTIQMLVALKPIRDAVGIERINVCTYQAVSGTGKEAIEELATQTANLLNAKDIKCEVYPKQIA...
  output: A summary of the protein's main attributes with the input amino acid sequence reveals: Catalyzes the NADPH-dependent formation of L-aspartate- semialdehyde (L-ASA) by the reductive dephosphorylation o...
  metadata: {'protein_accession': 'A0A849MGI6', 'num_tokens_in_input': 349, 'seq_len': 341, 'task': 'general_function', 'annots': 'Catalyzes the NADPH-dependent formation of L-aspartate- semialdehyde (L-ASA) by the reductive dephosphorylation of L-aspartyl- 4-phosphate.', 'split': 'train'}

Task: protein_function
Fields: ['instruction', 'input', 'output', 'metad

In [16]:
# Show more detailed examples for each task
print("\n" + "=" * 80)
print("DETAILED EXAMPLES FROM EACH TASK")
print("=" * 80)

for task_name, samples in mol_instructions_data.items():
    print(f"\n{'='*60}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*60}")
    
    for i, sample in enumerate(samples[:3]):
        print(f"\n--- Example {i+1} ---")
        if 'instruction' in sample:
            print(f"INSTRUCTION: {sample['instruction'][:300]}..." if len(sample.get('instruction', '')) > 300 else f"INSTRUCTION: {sample.get('instruction', 'N/A')}")
        if 'input' in sample:
            input_text = sample['input']
            print(f"INPUT: {input_text[:300]}..." if len(input_text) > 300 else f"INPUT: {input_text}")
        if 'output' in sample:
            output_text = sample['output']
            print(f"OUTPUT: {output_text[:300]}..." if len(output_text) > 300 else f"OUTPUT: {output_text}")


DETAILED EXAMPLES FROM EACH TASK

TASK: GENERAL_FUNCTION

--- Example 1 ---
INSTRUCTION: Inspect the protein with the subsequent sequence and offer a concise description of its properties: 
INPUT: ```
MTDKKFNVAVVGATGAVGETMLSILEQRHFPVGEVYPLASSRSAGKRIEFNNKQLIVEDLETFDFSKVQIGLFSPGASVSAIHAPRAVEAGCVVVDNTSQFRYDDDIPLVVSEVNPHAVADYKNRGIIANPNCSTIQMLVALKPIRDAVGIERINVCTYQAVSGTGKEAIEELATQTANLLNAKDIKCEVYPKQIAFNALPQIDVFQENGYTKEEMKMVWETRKIFEDESILVNPTAVRIPVFYGHSEAVHIETKEKIDVATARKLLEEMPGVTVLDERQDGGYPTAVTEAAGEDP...
OUTPUT: A summary of the protein's main attributes with the input amino acid sequence reveals: Catalyzes the NADPH-dependent formation of L-aspartate- semialdehyde (L-ASA) by the reductive dephosphorylation of L-aspartyl- 4-phosphate.

--- Example 2 ---
INSTRUCTION: Please provide a summary of the key features and characteristics of the protein with the following amino acid sequence: 
INPUT: ```
MHQQSPVDDVTALNSSALTMSEYPEGESPLQLQDVDSSRVGGHILSPIFNSSSPSLPVESHPVCIQSPYTDLGHDFTTLPFYSPALLGYGTSPLSECSS

In [17]:
# Summary statistics
print("\n" + "=" * 60)
print("MOL-INSTRUCTIONS SUMMARY")
print("=" * 60)

for task_name, samples in mol_instructions_data.items():
    # Get actual file size to estimate total entries
    json_file = mol_inst_dir / f"{task_name}.json"
    with open(json_file, 'r') as f:
        full_data = json.load(f)
    
    total_entries = len(full_data) if isinstance(full_data, list) else 1
    
    # Calculate average lengths
    if samples and 'input' in samples[0]:
        avg_input_len = sum(len(s.get('input', '')) for s in samples) / len(samples)
        avg_output_len = sum(len(s.get('output', '')) for s in samples) / len(samples)
        print(f"\n{task_name}:")
        print(f"  Total entries: {total_entries:,}")
        print(f"  Avg input length: {avg_input_len:.0f} chars")
        print(f"  Avg output length: {avg_output_len:.0f} chars")


MOL-INSTRUCTIONS SUMMARY



general_function:
  Total entries: 86,572
  Avg input length: 415 chars
  Avg output length: 349 chars



protein_function:
  Total entries: 114,183
  Avg input length: 384 chars
  Avg output length: 323 chars



domain_motif:
  Total entries: 45,100
  Avg input length: 485 chars
  Avg output length: 155 chars



catalytic_activity:
  Total entries: 53,174
  Avg input length: 440 chars
  Avg output length: 219 chars



protein_design:
  Total entries: 195,975
  Avg input length: 571 chars
  Avg output length: 443 chars


---
## Summary

### Dataset Overview

| Dataset | Location | Format | Size |
|---------|----------|--------|------|
| IPD PDB Sample | `data/raw/pdb_2021aug02_sample` | PyTorch .pt files | ~870 structures |
| Swiss-Prot | `data/raw/swissprot/uniprot_sprot.fasta.gz` | Gzipped FASTA | ~570K sequences |
| Mol-Instructions | `data/raw/mol_instructions/data/Protein-oriented_Instructions/` | JSON files | ~500K instruction pairs |

In [18]:
print("Dataset exploration complete!")

Dataset exploration complete!
